In [1]:
!pip install numpy pillow scipy matplotlib

In [2]:
from pathlib import Path
import numpy as np
import urllib.request
from PIL import Image, ImageDraw, ImageFont
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [3]:
from_text = "TEXTFLOW"
to_text = "F10WTEXT"

In [4]:
# Download the font if it doesn't exist

font_url="https://github.com/googlefonts/noto-cjk/raw/main/Sans/OTF/Japanese/NotoSansCJKjp-Regular.otf"
font_path = Path(font_url).name
font_file = Path(font_path)

if not font_file.exists():
    print(f"Downloading font from {font_url} to {font_path}")
    urllib.request.urlretrieve(font_url, font_path)
else:
    print(f"Font already exists: {font_path}")

In [5]:
def load_font(font_path):
    try:
        font = ImageFont.truetype(font_path, size=60)
    except:
        font = ImageFont.load_default()
        print("Could not load font, using default.")
    return font

def compute_bbox(text, font_path):
    font = load_font(font_path)

    bbox = font.getbbox(text)
    return [int(x) for x in bbox]


def text_to_points(text, font_path, bbox):
    """
    Convert a text string into an array of 2D points.
    """
    font = load_font(font_path)

    left, top, right, bottom = [int(x) for x in bbox]
    width = right - left
    height = bottom - top
    img = Image.new("L", (width, height), color=0)
    draw = ImageDraw.Draw(img)
    draw.text((-left, -top), text, font=font, fill="white")

    arr = np.array(img)
    y_idxs, x_idxs = np.nonzero(arr)

    pts = np.stack([x_idxs, y_idxs], axis=1).astype(np.float32)

    pts -= pts.mean(axis=0)
    pts /= max(width, height) / 2  # scale to fit in [-1,1]

    return pts


def upsample_points(points, target_n):
    """
    Upsample the point set while keeping original points.
    """
    n = len(points)
    
    if target_n <= n:
        return points
    
    extra = target_n - n
    idx = np.random.choice(n, extra, replace=True)
    
    extra_points = points[idx]
    
    return np.vstack([points, extra_points])


def balance_point_sets(src, tgt):
    n_points_src = len(src)
    n_points_tgt = len(tgt)

    if n_points_src > n_points_tgt:
        tgt = upsample_points(tgt, n_points_src)
    elif n_points_tgt > n_points_src:
        src = upsample_points(src, n_points_tgt)

    return src, tgt

In [6]:
from_bbox = compute_bbox(from_text, font_path)
to_bbox = compute_bbox(to_text, font_path)
max_bbox = [
    min(from_bbox[0], to_bbox[0]),
    min(from_bbox[1], to_bbox[1]),
    max(from_bbox[2], to_bbox[2]),
    max(from_bbox[3], to_bbox[3]),
]
width = max_bbox[2] - max_bbox[0]
height = max_bbox[3] - max_bbox[1]

# Render text as points on the same canvas size for both texts
src = text_to_points(from_text,font_path, max_bbox)
tgt = text_to_points(to_text, font_path, max_bbox)


src, tgt = balance_point_sets(src, tgt)  # make point sets the same size
print("Number of points used:", len(src))

Number of points used: 4506


In [7]:
C = cdist(src, tgt, metric="sqeuclidean")
row_ind, col_ind = linear_sum_assignment(C)
P = np.zeros((len(src), len(src)))
P[row_ind, col_ind] = 1 / len(src)  # normalize

In [8]:
fps = 30
move_frames = 60
hold_frames = 30

x = np.linspace(0, 1, move_frames, endpoint=False)
ease_up = (1 - np.cos(np.pi * x)) / 2
ease_down = ease_up[::-1]  # reverse

# Allows looping with a hold at two ends of the animation
t = np.concatenate([np.zeros(hold_frames), ease_up, np.ones(hold_frames), ease_down])

In [9]:
frames = []

for alpha in t:
    frame_points = (1 - alpha) * src + alpha * tgt[col_ind]  # linear interpolation
    frames.append(frame_points)

In [10]:
base_fig_width = 7
width_ratio = width / height

fig, ax = plt.subplots(figsize=(base_fig_width, base_fig_width / width_ratio))
fig.patch.set_facecolor("black")
sc = ax.scatter(src[:, 0], src[:, 1], s=1, c="white")

pad = 0.05

ax.set_xlim(-1 - pad, 1 + pad)
ax.set_ylim(
    (-1 / width_ratio) - pad,
    (1 / width_ratio) + pad
) # scale y to keep points proportional
ax.set_aspect("equal")  # preserve ratio

ax.invert_yaxis()
ax.axis("off")


def update(frame_points):
    sc.set_offsets(frame_points)
    return (sc,)


ani = FuncAnimation(fig, update, frames=frames, interval=1e3/fps, blit=True)
plt.close(fig)

HTML(ani.to_jshtml())

In [11]:
# Save the animation as a webm video

ani.save(
    "text_anim.webm",
    writer="ffmpeg",
    fps=fps,
    codec="libvpx-vp9",
    bitrate=2000
)